# SASRec fixed notebook

This notebook fixes the previous SASRec collapse where almost every user received the same 10 items.

Main changes:

1. **Many negatives per positive** using sampled-softmax / cross-entropy instead of one BCE negative.
2. **Cosine scoring option at inference** to remove item-norm artefacts that caused rare high-norm items to dominate.
3. **Validation grid over inference scoring and popularity blending** before final submission.
4. **Submission diversity diagnostics** so collapse is caught immediately.
5. Unique output directory per run, so it can run simultaneously with other notebooks.

In [11]:
from pathlib import Path
from datetime import datetime, timezone
import uuid
import math
import random
import json
import time
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_DIR = Path("data/")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
OUTPUT_DIR = Path("outputs") / f"sasrec_fixed_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DATA_DIR =", DATA_DIR.resolve())
print("OUTPUT_DIR =", OUTPUT_DIR.resolve())
print("DEVICE =", DEVICE)

DATA_DIR = /home/joris/Master/Semester 2/Recommender Systems/Final_Assignment_RS/data
OUTPUT_DIR = /home/joris/Master/Semester 2/Recommender Systems/Final_Assignment_RS/outputs/sasrec_fixed_20260609_191012_e7519c
DEVICE = cuda


In [12]:
# Configuration.
# For CPU-only runs, lower num_negatives to 30-50 and/or hidden_units to 64.
FAST_DEV_RUN = False
RUN_VALIDATION = True
RUN_FINAL_TRAINING = True

CONFIG = {
    "max_len": 50,
    "hidden_units": 512,
    "num_blocks": 4,
    "num_heads": 2,
    "dropout_rate": 0.2,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "batch_size": 256,
    "epochs": 3 if FAST_DEV_RUN else 100,
    "patience": 20,
    "num_workers": 0,

    # Main fix: one negative per position is too weak for ~13k items.
    # 50-100 is usually a much better starting range for this sparse catalog.
    "num_negatives": 100,

    # Main fix: cosine scoring prevents rare high-norm item embeddings from dominating inference.
    # Validation grid below will compare dot vs cosine and tune popularity alpha.
    "score_mode": "cosine",

    # This is tuned in validation grid; keep small-to-moderate.
    "popularity_blend_alpha": 0.05,
}
CONFIG

{'max_len': 50,
 'hidden_units': 512,
 'num_blocks': 4,
 'num_heads': 2,
 'dropout_rate': 0.2,
 'lr': 0.001,
 'weight_decay': 1e-05,
 'batch_size': 256,
 'epochs': 100,
 'patience': 20,
 'num_workers': 0,
 'num_negatives': 100,
 'score_mode': 'cosine',
 'popularity_blend_alpha': 0.05}

In [13]:
# Load and de-duplicate exact user-item events.
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

train_all = (
    train_raw
    .sort_values(["timestamp", "user_id", "item_id"])
    .drop_duplicates(["user_id", "item_id"], keep="first")
    .copy()
)

cutoff = test_raw["timestamp"].min()
train_fit = train_all[train_all["timestamp"] < cutoff].copy()
valid = train_all[train_all["timestamp"] >= cutoff].copy()

print("train_raw:", train_raw.shape)
print("train_all dedup:", train_all.shape)
print("train_fit:", train_fit.shape)
print("valid:", valid.shape)
print("sample users:", sample["user_id"].nunique())

display(train_all.head())

train_raw: (162727, 3)
train_all dedup: (158471, 3)
train_fit: (143313, 3)
valid: (15158, 3)
sample users: 2255


,item_id,user_id,timestamp
0,7926,22119,1020909887000
1,6719,3664,1072719074000
2,6719,5466,1090094915000
3,6719,16280,1152498722000
4,9677,6383,1161911421000


In [14]:
# Quick diagnostic for an existing collapsed SASRec submission, if present.
maybe_bad = DATA_DIR / "submission_sasrec.csv"
if maybe_bad.exists():
    bad = pd.read_csv(maybe_bad)
    print("Existing submission rows:", len(bad))
    print("Unique recommendation strings:", bad["item_id"].nunique())
    display(bad["item_id"].value_counts().head(10).rename("count").reset_index())

In [15]:
class PointWiseFeedForward(nn.Module):
    def __init__(self, hidden_units, dropout_rate):
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout1 = nn.Dropout(p=dropout_rate)
        self.conv2 = nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout2 = nn.Dropout(p=dropout_rate)

    def forward(self, inputs):
        outputs = self.dropout2(self.conv2(F.relu(self.dropout1(self.conv1(inputs.transpose(-1, -2))))))
        return outputs.transpose(-1, -2)


class SASRec(nn.Module):
    # Copied/adapted from the previous assignment implementation.
    # The model architecture is mostly unchanged; the training objective below is what is fixed.
    def __init__(
        self,
        user_num,
        item_num,
        device,
        norm_first=False,
        maxlen=50,
        dropout_rate=0.2,
        hidden_units=128,
        num_heads=2,
        num_blocks=3,
    ):
        super().__init__()
        self.user_num = user_num
        self.item_num = item_num
        self.dev = device
        self.norm_first = norm_first
        self.maxlen = maxlen

        self.item_emb = nn.Embedding(self.item_num + 1, hidden_units, padding_idx=0)
        self.pos_emb = nn.Embedding(maxlen + 1, hidden_units, padding_idx=0)
        self.emb_dropout = nn.Dropout(p=dropout_rate)

        self.attention_layernorms = nn.ModuleList()
        self.attention_layers = nn.ModuleList()
        self.forward_layernorms = nn.ModuleList()
        self.forward_layers = nn.ModuleList()
        self.last_layernorm = nn.LayerNorm(hidden_units, eps=1e-8)

        for _ in range(num_blocks):
            self.attention_layernorms.append(nn.LayerNorm(hidden_units, eps=1e-8))
            self.attention_layers.append(nn.MultiheadAttention(hidden_units, num_heads, dropout_rate))
            self.forward_layernorms.append(nn.LayerNorm(hidden_units, eps=1e-8))
            self.forward_layers.append(PointWiseFeedForward(hidden_units, dropout_rate))

        for _, param in self.named_parameters():
            try:
                torch.nn.init.xavier_normal_(param.data)
            except Exception:
                pass

        with torch.no_grad():
            self.item_emb.weight[0].fill_(0)
            self.pos_emb.weight[0].fill_(0)

    def log2feats(self, log_seqs):
        log_seqs = log_seqs.long().to(self.dev)
        seqs = self.item_emb(log_seqs)
        seqs *= self.item_emb.embedding_dim ** 0.5

        # Absolute positions, zero for padding.
        poss = torch.arange(1, log_seqs.shape[1] + 1, device=self.dev).unsqueeze(0).expand(log_seqs.shape[0], -1)
        poss = poss * (log_seqs != 0)
        seqs = seqs + self.pos_emb(poss.long())
        seqs = self.emb_dropout(seqs)

        tl = seqs.shape[1]
        causal_mask = ~torch.tril(torch.ones((tl, tl), dtype=torch.bool, device=self.dev))

        # Padding mask is important on this sparse data, where most rows are mostly padding.
        # True means "ignore this key position".
        key_padding_mask = (log_seqs == 0)

        for i in range(len(self.attention_layers)):
            seqs = torch.transpose(seqs, 0, 1)  # L, B, H

            if self.norm_first:
                x = self.attention_layernorms[i](seqs)
                mha_outputs, _ = self.attention_layers[i](
                    x, x, x,
                    attn_mask=causal_mask,
                    key_padding_mask=key_padding_mask,
                    need_weights=False,
                )
                seqs = seqs + mha_outputs
                seqs = torch.transpose(seqs, 0, 1)
                seqs = seqs + self.forward_layers[i](self.forward_layernorms[i](seqs))
            else:
                mha_outputs, _ = self.attention_layers[i](
                    seqs, seqs, seqs,
                    attn_mask=causal_mask,
                    key_padding_mask=key_padding_mask,
                    need_weights=False,
                )
                seqs = self.attention_layernorms[i](seqs + mha_outputs)
                seqs = torch.transpose(seqs, 0, 1)
                seqs = self.forward_layernorms[i](seqs + self.forward_layers[i](seqs))

        return self.last_layernorm(seqs)

    def predict(self, user_ids, log_seqs, item_indices, score_mode="dot"):
        log_feats = self.log2feats(log_seqs)
        final_feat = log_feats[:, -1, :]
        item_embs = self.item_emb(item_indices.long().to(self.dev))

        if score_mode == "cosine":
            final_feat = F.normalize(final_feat, dim=-1)
            item_embs = F.normalize(item_embs, dim=-1)
        elif score_mode != "dot":
            raise ValueError("score_mode must be 'dot' or 'cosine'")

        logits = item_embs.matmul(final_feat.unsqueeze(-1)).squeeze(-1)
        return logits

In [16]:
class KaggleSASRecDataset(Dataset):
    """
    SASRec dataset for timestamped implicit feedback.

    Items are mapped to 1..num_items; 0 is padding.
    Each user contributes one training row containing all prefix->next-item positions.
    Negatives are sampled dynamically inside the training loop, so this dataset returns only seq and pos.
    """
    def __init__(self, interactions: pd.DataFrame, max_len=50, min_train_len=2):
        self.max_len = max_len
        self.min_train_len = min_train_len

        interactions = interactions.sort_values(["user_id", "timestamp", "item_id"]).copy()
        self.raw_users = np.sort(interactions["user_id"].unique())
        self.raw_items = np.sort(interactions["item_id"].unique())

        self.user2id = {int(u): idx for idx, u in enumerate(self.raw_users)}
        self.id2user = {idx: int(u) for u, idx in self.user2id.items()}
        self.item2id = {int(it): idx + 1 for idx, it in enumerate(self.raw_items)}
        self.id2item = {idx: int(it) for it, idx in self.item2id.items()}

        self.num_users = len(self.user2id)
        self.num_items = len(self.item2id)

        self.user_sequences = {}
        self.user_pos_items = {}

        for raw_user, grp in interactions.groupby("user_id", sort=False):
            u = self.user2id[int(raw_user)]
            seq = [self.item2id[int(it)] for it in grp.sort_values(["timestamp", "item_id"])["item_id"].tolist()]
            self.user_sequences[u] = seq
            self.user_pos_items[u] = set(seq)

        self.train_data = []
        self._build_train_data()

    def _make_arrays(self, train_seq):
        seq = [0] * self.max_len
        pos = [0] * self.max_len

        nxt = train_seq[-1]
        idx = self.max_len - 1

        # seq[t] predicts pos[t].
        for item in reversed(train_seq[:-1]):
            seq[idx] = item
            pos[idx] = nxt
            nxt = item
            idx -= 1
            if idx < 0:
                break

        return seq, pos

    def _build_train_data(self):
        self.train_data = []
        for user, train_seq in self.user_sequences.items():
            if len(train_seq) < self.min_train_len:
                continue
            seq, pos = self._make_arrays(train_seq)
            self.train_data.append((user, seq, pos))

    def _pad_sequence(self, seq):
        seq = seq[-self.max_len:]
        return [0] * (self.max_len - len(seq)) + seq

    def sequence_for_raw_user(self, raw_user):
        raw_user = int(raw_user)
        if raw_user not in self.user2id:
            return [0] * self.max_len
        return self._pad_sequence(self.user_sequences[self.user2id[raw_user]])

    def map_validation_targets(self, valid_df: pd.DataFrame):
        tmp = valid_df[["user_id", "item_id"]].copy()
        tmp["u"] = tmp["user_id"].map(self.user2id)
        tmp["i"] = tmp["item_id"].map(self.item2id)
        tmp = tmp.dropna(subset=["u", "i"]).astype({"u": "int64", "i": "int64"})
        return tmp.groupby("u")["i"].apply(set).to_dict()

    def eval_prefixes_for_targets(self, targets):
        rows = []
        for u in sorted(targets.keys()):
            rows.append((u, self._pad_sequence(self.user_sequences[u]), targets[u]))
        return rows

    def __len__(self):
        return len(self.train_data)

    def __getitem__(self, idx):
        user, seq, pos = self.train_data[idx]
        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(seq, dtype=torch.long),
            torch.tensor(pos, dtype=torch.long),
        )

In [17]:
def sampled_softmax_loss(model, log_seqs, pos_seqs, num_items, num_negatives):
    """
    Multi-negative next-item loss.

    This replaces the weak one-negative BCE loss. For each non-padding target position,
    the model must rank the true next item above K random negatives.
    """
    log_feats = model.log2feats(log_seqs)
    mask = pos_seqs > 0

    if not mask.any():
        return torch.tensor(0.0, device=log_seqs.device, requires_grad=True)

    feats = log_feats[mask]             # N, H
    pos_items = pos_seqs[mask].long()   # N
    n_targets = pos_items.shape[0]

    # Dynamic negative sampling. Collision with a positive target is repaired.
    neg_items = torch.randint(
        low=1,
        high=num_items + 1,
        size=(n_targets, num_negatives),
        device=feats.device,
    )
    neg_items = torch.where(
        neg_items == pos_items.unsqueeze(1),
        (neg_items % num_items) + 1,
        neg_items,
    )

    pos_emb = model.item_emb(pos_items)           # N, H
    neg_emb = model.item_emb(neg_items)           # N, K, H

    pos_logits = (feats * pos_emb).sum(dim=-1, keepdim=True)  # N, 1
    neg_logits = torch.bmm(neg_emb, feats.unsqueeze(-1)).squeeze(-1)  # N, K

    logits = torch.cat([pos_logits, neg_logits], dim=1)
    labels = torch.zeros(n_targets, dtype=torch.long, device=feats.device)

    return F.cross_entropy(logits, labels)


def make_popularity_scores(dataset):
    counts = np.zeros(dataset.num_items + 1, dtype=np.float32)
    for seq in dataset.user_sequences.values():
        for item in seq:
            counts[item] += 1
    scores = np.log1p(counts[1:])
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def standardize_rows(scores):
    # Helps blend dot/cosine scores with popularity in a stable way.
    mu = scores.mean(dim=1, keepdim=True)
    sd = scores.std(dim=1, keepdim=True).clamp_min(1e-6)
    return (scores - mu) / sd

In [18]:
@torch.no_grad()
def evaluate_sasrec(
    model,
    dataset: KaggleSASRecDataset,
    valid_df,
    k=10,
    batch_size=256,
    popularity_scores=None,
    alpha=0.0,
    score_mode="cosine",
    standardize_model_scores=True,
):
    model.eval()
    targets = dataset.map_validation_targets(valid_df)
    eval_rows = dataset.eval_prefixes_for_targets(targets)
    recalls, ndcgs = [], []

    all_items = torch.arange(dataset.num_items + 1, device=DEVICE)
    pop = torch.tensor(popularity_scores, dtype=torch.float32, device=DEVICE) if popularity_scores is not None else None

    for start in range(0, len(eval_rows), batch_size):
        batch = eval_rows[start:start + batch_size]
        users = [r[0] for r in batch]
        seqs = [r[1] for r in batch]
        true_sets = [r[2] for r in batch]

        user_tensor = torch.tensor(users, dtype=torch.long, device=DEVICE)
        seq_tensor = torch.tensor(seqs, dtype=torch.long, device=DEVICE)

        scores = model.predict(user_tensor, seq_tensor, all_items, score_mode=score_mode)
        scores[:, 0] = -float("inf")

        if standardize_model_scores:
            finite_scores = scores[:, 1:]
            scores[:, 1:] = standardize_rows(finite_scores)

        if pop is not None and alpha > 0:
            scores[:, 1:] = scores[:, 1:] + alpha * pop.unsqueeze(0)

        # Exclude items seen in the prefix.
        for row, u in enumerate(users):
            seen = dataset.user_pos_items.get(u, set())
            if seen:
                scores[row, torch.tensor(list(seen), dtype=torch.long, device=DEVICE)] = -float("inf")

        topk = torch.topk(scores, k=k, dim=1).indices.cpu().numpy()

        for pred, true_items in zip(topk, true_sets):
            pred = [int(x) for x in pred]
            hits = [item for item in pred if item in true_items]
            recalls.append(len(hits) / min(k, len(true_items)))
            if hits:
                dcg = sum(1.0 / math.log2(pred.index(item) + 2) for item in hits)
                ideal = sum(1.0 / math.log2(r + 2) for r in range(min(k, len(true_items))))
                ndcgs.append(dcg / ideal)
            else:
                ndcgs.append(0.0)

    return {
        "Recall@10": float(np.mean(recalls)) if recalls else 0.0,
        "NDCG@10": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "num_eval_users": len(eval_rows),
        "score_mode": score_mode,
        "alpha": alpha,
    }

In [19]:
def train_sasrec(train_df, valid_df=None, config=CONFIG):
    dataset = KaggleSASRecDataset(train_df, max_len=config["max_len"])
    model = SASRec(
        user_num=dataset.num_users,
        item_num=dataset.num_items,
        device=DEVICE,
        maxlen=dataset.max_len,
        hidden_units=config["hidden_units"],
        num_heads=config["num_heads"],
        num_blocks=config["num_blocks"],
        dropout_rate=config["dropout_rate"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        betas=(0.9, 0.98),
        weight_decay=config["weight_decay"],
    )

    pop_scores = make_popularity_scores(dataset)

    best_score = -1.0
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, config["epochs"] + 1):
        loader = DataLoader(
            dataset,
            batch_size=config["batch_size"],
            shuffle=True,
            num_workers=config["num_workers"],
            pin_memory=(DEVICE.type == "cuda"),
        )

        model.train()
        total_loss = 0.0
        n_rows = 0
        t0 = time.time()

        for user_ids, log_seqs, pos_seqs in loader:
            log_seqs = log_seqs.to(DEVICE, non_blocking=True)
            pos_seqs = pos_seqs.to(DEVICE, non_blocking=True)

            loss = sampled_softmax_loss(
                model,
                log_seqs,
                pos_seqs,
                num_items=dataset.num_items,
                num_negatives=config["num_negatives"],
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            bs = log_seqs.shape[0]
            total_loss += loss.item() * bs
            n_rows += bs

        row = {
            "epoch": epoch,
            "loss": total_loss / max(n_rows, 1),
            "seconds": time.time() - t0,
        }

        if valid_df is not None:
            # Use chosen config for early stopping.
            metrics = evaluate_sasrec(
                model,
                dataset,
                valid_df,
                k=10,
                popularity_scores=pop_scores,
                alpha=config["popularity_blend_alpha"],
                score_mode=config["score_mode"],
            )
            row.update(metrics)

            print(
                f"epoch {epoch:03d} loss={row['loss']:.5f} "
                f"recall10={metrics['Recall@10']:.5f} ndcg10={metrics['NDCG@10']:.5f} "
                f"mode={metrics['score_mode']} alpha={metrics['alpha']} "
                f"time={row['seconds']:.1f}s"
            )

            score = metrics["Recall@10"]
            if score > best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= config["patience"]:
                    print("early stopping")
                    break
        else:
            print(f"epoch {epoch:03d} loss={row['loss']:.5f} time={row['seconds']:.1f}s")

        history.append(row)

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, dataset, pop_scores, pd.DataFrame(history)

In [20]:
if RUN_VALIDATION:
    sas_model, sas_dataset, sas_pop_scores, sas_history = train_sasrec(train_fit, valid, CONFIG)
    sas_history.to_csv(OUTPUT_DIR / "sasrec_fixed_validation_history.csv", index=False)
    display(sas_history.tail())

    # Tune only inference scoring/blending without retraining.
    grid_rows = []
    for score_mode in ["cosine", "dot"]:
        for alpha in [0.0, 0.02, 0.05, 0.10, 0.20]:
            m = evaluate_sasrec(
                sas_model,
                sas_dataset,
                valid,
                k=10,
                popularity_scores=sas_pop_scores,
                alpha=alpha,
                score_mode=score_mode,
            )
            grid_rows.append(m)

    grid = pd.DataFrame(grid_rows).sort_values(["Recall@10", "NDCG@10"], ascending=False)
    display(grid)
    grid.to_csv(OUTPUT_DIR / "sasrec_fixed_inference_grid.csv", index=False)

    best = grid.iloc[0].to_dict()
    CONFIG["score_mode"] = best["score_mode"]
    CONFIG["popularity_blend_alpha"] = float(best["alpha"])
    print("Best inference settings:", {"score_mode": CONFIG["score_mode"], "alpha": CONFIG["popularity_blend_alpha"]})

    with open(OUTPUT_DIR / "sasrec_fixed_config_after_validation.json", "w") as f:
        json.dump(CONFIG, f, indent=2)

epoch 001 loss=4.37352 recall10=0.00476 ndcg10=0.00221 mode=cosine alpha=0.05 time=4.2s
epoch 002 loss=4.22834 recall10=0.00819 ndcg10=0.00509 mode=cosine alpha=0.05 time=4.0s
epoch 003 loss=4.21033 recall10=0.00906 ndcg10=0.00561 mode=cosine alpha=0.05 time=4.0s
epoch 004 loss=4.19507 recall10=0.00821 ndcg10=0.00545 mode=cosine alpha=0.05 time=3.9s
epoch 005 loss=4.18987 recall10=0.00970 ndcg10=0.00637 mode=cosine alpha=0.05 time=3.9s
epoch 006 loss=4.18318 recall10=0.00797 ndcg10=0.00545 mode=cosine alpha=0.05 time=3.8s
epoch 007 loss=4.17970 recall10=0.00947 ndcg10=0.00587 mode=cosine alpha=0.05 time=3.9s
epoch 008 loss=4.17027 recall10=0.00890 ndcg10=0.00579 mode=cosine alpha=0.05 time=4.0s
epoch 009 loss=4.15973 recall10=0.01028 ndcg10=0.00624 mode=cosine alpha=0.05 time=3.9s
epoch 010 loss=4.14795 recall10=0.01151 ndcg10=0.00707 mode=cosine alpha=0.05 time=3.9s
epoch 011 loss=4.13834 recall10=0.01153 ndcg10=0.00665 mode=cosine alpha=0.05 time=4.0s
epoch 012 loss=4.14202 recall10=

,epoch,loss,seconds,Recall@10,NDCG@10,num_eval_users,score_mode,alpha
44,45,3.636766,4.036000,0.011090,0.006966,7437,cosine,0.05
45,46,3.654353,4.058674,0.011809,0.006993,7437,cosine,0.05
46,47,3.636150,4.020884,0.012351,0.007383,7437,cosine,0.05
47,48,3.601697,3.974983,0.010972,0.006769,7437,cosine,0.05
48,49,3.601819,3.864742,0.012175,0.007441,7437,cosine,0.05


,Recall@10,NDCG@10,num_eval_users,score_mode,alpha
4,0.014046,0.008133,7437,cosine,0.20
3,0.013997,0.008100,7437,cosine,0.10
2,0.013747,0.008099,7437,cosine,0.05
0,0.013703,0.008070,7437,cosine,0.00
1,0.013658,0.008051,7437,cosine,0.02
9,0.013348,0.008125,7437,dot,0.20
5,0.013341,0.008129,7437,dot,0.00
6,0.013296,0.008111,7437,dot,0.02
8,0.013274,0.008104,7437,dot,0.10
7,0.013229,0.008090,7437,dot,0.05


Best inference settings: {'score_mode': 'cosine', 'alpha': 0.2}


In [21]:
@torch.no_grad()
def recommend_sasrec(
    model,
    dataset: KaggleSASRecDataset,
    raw_users,
    k=10,
    popularity_scores=None,
    alpha=0.0,
    batch_size=256,
    score_mode="cosine",
    standardize_model_scores=True,
):
    model.eval()

    all_items = torch.arange(dataset.num_items + 1, device=DEVICE)
    pop = torch.tensor(popularity_scores, dtype=torch.float32, device=DEVICE) if popularity_scores is not None else None

    # Popularity fallback in raw item IDs.
    pop_internal = np.argsort(-(popularity_scores if popularity_scores is not None else np.ones(dataset.num_items))) + 1
    pop_rank_raw = [int(dataset.id2item[int(i)]) for i in pop_internal]

    recs = {}
    raw_users = [int(u) for u in raw_users]

    for start in range(0, len(raw_users), batch_size):
        batch_raw = raw_users[start:start + batch_size]
        mapped_users = [dataset.user2id.get(u, -1) for u in batch_raw]
        seqs = [dataset.sequence_for_raw_user(u) for u in batch_raw]

        user_tensor = torch.tensor([max(u, 0) for u in mapped_users], dtype=torch.long, device=DEVICE)
        seq_tensor = torch.tensor(seqs, dtype=torch.long, device=DEVICE)

        scores = model.predict(user_tensor, seq_tensor, all_items, score_mode=score_mode)
        scores[:, 0] = -float("inf")

        if standardize_model_scores:
            scores[:, 1:] = standardize_rows(scores[:, 1:])

        if pop is not None and alpha > 0:
            scores[:, 1:] = scores[:, 1:] + alpha * pop.unsqueeze(0)

        for row, u_mapped in enumerate(mapped_users):
            if u_mapped >= 0:
                seen = dataset.user_pos_items.get(u_mapped, set())
                if seen:
                    scores[row, torch.tensor(list(seen), dtype=torch.long, device=DEVICE)] = -float("inf")

        topk = torch.topk(scores, k=k, dim=1).indices.cpu().numpy()

        for raw_u, item_idxs, u_mapped in zip(batch_raw, topk, mapped_users):
            if u_mapped < 0:
                recs[raw_u] = pop_rank_raw[:k]
            else:
                rec = [int(dataset.id2item[int(i)]) for i in item_idxs if int(i) != 0][:k]
                if len(rec) < k:
                    already = set(rec)
                    rec += [it for it in pop_rank_raw if it not in already][: k - len(rec)]
                recs[raw_u] = rec

    return recs


def write_submission(recs, sample_df, path):
    out = sample_df[["ID", "user_id"]].copy()
    out["item_id"] = out["user_id"].map(lambda u: ",".join(map(str, recs[int(u)][:10])))

    assert out["item_id"].str.split(",").map(len).eq(10).all()
    out.to_csv(path, index=False)

    print("Saved:", path)
    print("Rows:", len(out))
    print("Unique recommendation strings:", out["item_id"].nunique())
    display(out["item_id"].value_counts().head(10).rename("count").reset_index())
    display(out.head())

    if out["item_id"].nunique() < max(50, int(0.05 * len(out))):
        print("WARNING: recommendation diversity is still very low. Do not submit before inspecting validation metrics.")

    return out

In [22]:
if RUN_FINAL_TRAINING:
    final_sas_model, final_sas_dataset, final_sas_pop, final_sas_history = train_sasrec(train_all, valid_df=None, config=CONFIG)
    final_sas_history.to_csv(OUTPUT_DIR / "sasrec_fixed_final_training_history.csv", index=False)
    torch.save(final_sas_model.state_dict(), OUTPUT_DIR / "sasrec_fixed_final_model.pt")

    sas_recs = recommend_sasrec(
        final_sas_model,
        final_sas_dataset,
        raw_users=sample["user_id"].astype(int).tolist(),
        k=10,
        popularity_scores=final_sas_pop,
        alpha=CONFIG["popularity_blend_alpha"],
        score_mode=CONFIG["score_mode"],
    )

    submission = write_submission(sas_recs, sample, OUTPUT_DIR / "submission_sasrec_fixed.csv")
else:
    print("Set RUN_FINAL_TRAINING=True to train on all interactions and create final submission.")

epoch 001 loss=4.39126 time=4.2s
epoch 002 loss=4.23854 time=4.1s
epoch 003 loss=4.09571 time=4.2s
epoch 004 loss=4.03415 time=4.1s
epoch 005 loss=4.00948 time=4.0s
epoch 006 loss=3.98904 time=4.0s
epoch 007 loss=3.99098 time=4.0s
epoch 008 loss=3.97918 time=4.0s
epoch 009 loss=3.97900 time=4.0s
epoch 010 loss=3.96718 time=4.0s
epoch 011 loss=3.95087 time=4.0s
epoch 012 loss=3.96213 time=4.0s
epoch 013 loss=3.94267 time=4.0s
epoch 014 loss=3.95504 time=4.0s
epoch 015 loss=3.95175 time=4.0s
epoch 016 loss=3.92179 time=4.0s
epoch 017 loss=3.92913 time=4.0s
epoch 018 loss=3.91444 time=4.0s
epoch 019 loss=3.90135 time=4.0s
epoch 020 loss=3.85261 time=4.0s
epoch 021 loss=3.81352 time=4.0s
epoch 022 loss=3.78212 time=4.0s
epoch 023 loss=3.74294 time=4.0s
epoch 024 loss=3.71549 time=4.0s
epoch 025 loss=3.70028 time=4.2s
epoch 026 loss=3.68767 time=4.0s
epoch 027 loss=3.66584 time=4.0s
epoch 028 loss=3.64660 time=4.0s
epoch 029 loss=3.62778 time=4.0s
epoch 030 loss=3.60364 time=4.0s
epoch 031 

,item_id,count
0,"13296,6220,2343,10835,4413,3950,4388,8608,3428...",11
1,"2343,13296,6220,4413,3428,3950,544,5866,12665,...",9
2,"8703,7637,5149,12663,8981,9564,6474,5631,9668,...",7
3,"13296,2343,6220,4413,3950,10835,3428,544,4388,...",6
4,"13296,6220,2343,10835,4413,3950,4388,3428,8608...",6
5,"6220,13296,12470,11403,4413,3509,89,9410,5495,...",6
6,"2343,13296,6220,4413,3428,3950,544,5866,12665,...",5
7,"1290,7120,4929,10607,2916,10663,5135,296,789,3697",5
8,"7637,8703,8981,12663,5149,9564,6474,7706,11103...",5
9,"12793,4117,8922,5303,2140,8590,2990,4277,4315,...",5


,ID,user_id,item_id
0,12,12,"9799,8421,4798,2218,8413,6326,12026,6971,1996,..."
1,14,14,"6141,5149,544,11615,2343,8421,5642,9519,3950,8562"
2,17,17,"4158,12339,1328,840,4832,4080,12898,2004,9370,..."
3,21,21,"4798,7558,560,4117,6326,5495,3950,11289,10835,..."
4,44,44,"11298,5135,11024,4333,1290,5969,10389,9962,346..."


## Extra notes

If this fixed SASRec is still weaker than LightGCN, do not discard it immediately. Use it as an ensemble signal:

- average ranks from LightGCN and SASRec,
- or add SASRec score/rank as a feature in a reranker,
- or use SASRec only for users with longer chronological histories.

For this dataset, LightGCN can remain the main model because the graph signal appears strong; SASRec should mainly add sequence/recency diversity.